In [ ]:
# import numpy as np
# obs = np.load("results/obs_1.npy")
# print(obs.shape)
# import matplotlib.pyplot as plt
# for i in range(obs.shape[0]):
#     plt.imshow(obs[i, :, :, :3]/255.)
#     plt.show()
#     plt.imsave(f"results/obs_{i}.png", obs[i, :, :, :3]/255.)

In [ ]:
%reload_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

# check bev images
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import cv2

data_file = "/home/junzhewu/Projects/SG-VLN/robot_env/data/episode_data/grCommercial_scene0/bev_map_grCommercial_scene0.npz"
data = np.load(data_file)
# visualize the bev map with plotly
rgb, depth = data['rgb'], data['depth'][:,:,0]
depth[depth==np.inf] = 0
fig = px.imshow(rgb)
fig.show()

# fig = go.Figure(data=go.Heatmap(z=depth, colorscale='Viridis'))
fig = px.imshow(depth)
fig.show()

cv2.imwrite("rgb.png", rgb[...,[2,1,0,3]])


In [ ]:
%matplotlib inline
import sys
sys.path.append("..")
from matplotlib import pyplot as plt
from utils.astar import load_bev_map, generate_obstacle_map, AStarPlanner
from scipy.spatial.transform import Rotation as R
import json
import numpy as np

import cv2
scene_name = "grCommercial_scene1"
bev_map_path = f"../data/episode_data/{scene_name}/bev_map.npz"
bev_depth, info = load_bev_map(bev_map_path)
image_height, image_width = bev_depth.shape
obstacle_map = generate_obstacle_map(bev_depth)
px_per_meter = np.array(info["px_per_meter"])
world_center = np.array(info["world_center"])

# load a ref path
# load episode file first
episode_file = f"../episodes/{scene_name}.json"
with open(episode_file, 'r') as f:
    episode_config = json.load(f)

# load ref path
for episode_id in [3]:
# for episode_id in range(len(episode_config)):
    closest_goal_idx = episode_config[episode_id]["closest_goal_idx"]
    ref_path = episode_config[episode_id]["goals"][closest_goal_idx]["reference_path"]
    init_quat = episode_config[episode_id]["start_rotation"]
ref_waypoints = [[x[0], x[1], 0] for x in ref_path]

ref_waypoints_px = []
last_point = None
for i, point in enumerate(ref_path):
    point = np.array(point)
    if last_point is not None:
        if np.linalg.norm(point - last_point) < 0.5:
            continue
    last_point = point
    wp = (point[:2] - world_center[:2]) * px_per_meter
    ref_waypoints_px.append((wp[0]+image_width//2, image_height//2-wp[1], 0))


for i, point in enumerate(ref_waypoints_px):
    if i == len(ref_waypoints_px) - 1:
        yaw_angle = ref_waypoints_px[-2][2]
    else:
        yaw_angle = np.arctan2(ref_waypoints_px[i+1][1] - point[1], ref_waypoints_px[i+1][0] - point[0])
    ref_waypoints_px[i] = (ref_waypoints_px[i][0], ref_waypoints_px[i][1], yaw_angle)

fig = px.imshow(obstacle_map)


# connect waypoints with lines
fig.add_trace(go.Scatter(x=[point[0] for point in ref_waypoints_px], y=[point[1] for point in ref_waypoints_px], mode='markers+lines', marker=dict(size=10, color='red')))
fig.update_layout(width=500, height=500)
fig.show()

In [ ]:
"_".join("asdasd_asdasd_0".split("_")[:-1])

In [ ]:
# test one start and goal
astar = AStarPlanner(
    meters_per_cell=1/px_per_meter,
    robot_width=0.5, # 0.44
    robot_height=0.9, # 0.88
    step_size=0.2,
    step_size_yaw=45,
    reach_threshold=0.5,
)


start = [ref_waypoints_px[0][0], ref_waypoints_px[0][1], ref_waypoints_px[0][2]]
goal = [ref_waypoints_px[-1][0], ref_waypoints_px[-1][1], ref_waypoints_px[-1][2]]
# start = [450,942,0]
# start = [650,899,0]
# goal = [815,899,0]
start1 = astar.find_noncollide_neighbor(start, obstacle_map)


vis_points = [start, goal, start1]
vis_img = obstacle_map.copy()
for point in vis_points:
    vis_img, wall = astar.plot_rect(point, vis_img, 100, 3)
fig = plt.figure(figsize=(5, 5))
plt.imshow(vis_img, cmap='binary')
for x, y, yaw in vis_points:
    plt.quiver(x, y, np.cos(yaw), np.sin(yaw),
            angles='xy', scale_units='xy', scale=0.008, color='blue', width=0.01, label='Yaw Direction')
plt.show()


path = astar.plan(obstacle_map, start, goal, time_budget=5.0)

print(f"path: {np.array(path).shape}")
# path = astar.plan(start, goal)
vis_img = obstacle_map.copy()
for x, y, yaw in path:
    vis_img, wall = astar.plot_rect((x, y, yaw), vis_img, 100, 3)
fig = plt.figure(figsize=(10, 10))
plt.imshow(vis_img, cmap='binary')
if path:
    path = np.array(path)
    plt.scatter(path[:, 0], path[:, 1], color='red', s=3, label='Planned Path')
    plt.plot(path[:, 0], path[:, 1], 'r-', linewidth=1.5)
    plt.scatter(start[0], start[1], color='blue', s=50, marker='.', label='Start')
    plt.scatter(goal[0], goal[1], color='blue', s=50, marker='*', label='Goal')
    plt.legend()

    x,y,yaw = path[:,0], path[:,1], path[:,2]

    plt.quiver(x, y, np.cos(yaw), np.sin(yaw), np.arange(len(yaw)),
            angles='xy', scale_units='xy', scale=0.008, cmap='viridis', width=0.005, label='Yaw Direction')
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.legend()
    plt.axis("equal")
    plt.title("Waypoints with Yaw Directions in World Frame")
    plt.show()

In [ ]:
# test multiple start and goal
astar = AStarPlanner(
    meters_per_cell=1/px_per_meter,
    robot_width=0.5, # 0.44
    robot_height=0.9, # 0.88
    step_size=0.2,
    step_size_yaw=30,
    reach_threshold=0.5,
)

total_path = []
vis_points = []
start = ref_waypoints_px[0]
for i in range(len(ref_waypoints_px)-1):
    goal = ref_waypoints_px[i+1]
    vis_points.extend([start, goal])
    path = astar.plan(obstacle_map, start, goal, time_budget=1.0)
    total_path.extend(path)
    start = path[-1]
path = total_path

vis_img = obstacle_map.copy()
for point in vis_points:
    vis_img, wall = astar.plot_rect(point, vis_img, 100, 3)
fig = plt.figure(figsize=(5, 5))
plt.imshow(vis_img, cmap='binary')
for x, y, yaw in vis_points:
    plt.quiver(x, y, np.cos(yaw), np.sin(yaw),
            angles='xy', scale_units='xy', scale=0.008, color='blue', width=0.01, label='Yaw Direction')
plt.show()

print(f"path: {np.array(path).shape}")
# path = astar.plan(start, goal)
vis_img = obstacle_map.copy()
for x, y, yaw in path:
    vis_img, wall = astar.plot_rect((x, y, yaw), vis_img, 100, 3)
fig = plt.figure(figsize=(10, 10))
plt.imshow(vis_img, cmap='binary')
if path:
    path = np.array(path)
    plt.scatter(path[:, 0], path[:, 1], color='red', s=3, label='Planned Path')
    plt.plot(path[:, 0], path[:, 1], 'r-', linewidth=1.5)
    plt.scatter(start[0], start[1], color='blue', s=50, marker='.', label='Start')
    plt.scatter(goal[0], goal[1], color='blue', s=50, marker='*', label='Goal')
    plt.legend()

    x,y,yaw = path[:,0], path[:,1], path[:,2]

    plt.quiver(x, y, np.cos(yaw), np.sin(yaw), np.arange(len(yaw)),
            angles='xy', scale_units='xy', scale=0.008, cmap='viridis', width=0.005, label='Yaw Direction')
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.legend()
    plt.axis("equal")
    plt.title("Waypoints with Yaw Directions in World Frame")
    plt.show()

In [ ]:
astar.find_closest_yaw((0,0,0.1))

In [ ]:
node = (0,0,0.2)
yaw_list = [np.deg2rad(yaw) for yaw in range(-180, 180+1, astar.step_size_yaw)]
best_yaw = np.argmin([np.abs(astar.warp_to_pi(yaw - node[2])) for yaw in yaw_list])
yaw_list[best_yaw], np.rad2deg(node[2])

In [ ]:
yaw_list

In [ ]:
plt.imshow(np.zeros((100,100)), cmap='binary')
x,y,yaw = 50,50,np.deg2rad(45)
plt.quiver(x, y, np.cos(yaw), np.sin(yaw), 1,
        angles='xy', scale_units='xy', scale=0.05, color='white', width=0.01, label='Yaw Direction')
plt.show()